In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch14_웹 데이터 수집_1</font>
***

# 1. BeautifulSoup / parser
    (정적 웹크롤링, 공공 api 사용)

`pip install bs4` (아나콘다 설치 시 자동으로 설치되는 패키지에 포함됨)


▪ 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/ <br>
▪ Documentation : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

- requests 라이브러리
    - requests.get(url, headers=..., params=...): 서버에 페이지 요청을 보냄
    - headers: '크롬 브라우저'라고 신분 위장 (403/406 차단 방지)
    - params: 검색어, 페이지 번호 등 URL 쿼리 전달
    - response.status_code: 접속 응답 상태 확인 (200: 성공, 403/406: 차단, 404: 없음)
    - response.text: 서버가 넘겨준 웹페이지의 전체 HTML 원본 텍스트

In [ ]:
import requests # HTTP 요청을 처리하는 라이브러리

from requests_file import FileAdapter

In [ ]:
# 로컬 파일을 웹요청 하듯이 읽어오는 작업
s = requests.Session()
s.mount("file://", FileAdapter()) # file://로 시작하는 url을 어댑처로 처리
response = s.get("file:///ai/source/01_python/data/ch14_sample.html") # c:ai/source/01_python/data/ch14_sample.html
response

In [ ]:
if response:
    print('해당 url에 접근함')
else:
    print('url 접근 거부됨')

In [ ]:
response.status_code
# 200 : 정상, 404 : 없는 페이지

In [ ]:
response.content # html의 디코딩되지 않은 순수 바이너리(bytes) 데이터

In [ ]:
print(response.content.decode('utf-8'))

In [ ]:
response.text

In [ ]:
# HTML 파싱 객체
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

# soup

In [ ]:
# 1_soup.select_one('선택자') : 해당 선택자 중 첫 엘리먼트
el = soup.select_one('h1.greeting.css')

In [ ]:
print('el : ', el)
print('el.text : ', el.text)
print('el.string : ', el.string)
print('el의 속성들 :', el.attrs)
print('el의 클래스 속성 :', el.attrs['class'])
print('el의 클래스 속성 :', el.attrs.get('class'))
print('el의 href 속성 :', el.attrs.get('href')) # get() : 데이터가 없을 경우 None 반환
print('el의 이름 :', el.name)

In [ ]:
# 2_soup.select('선택자') : 해당 선택자 전체 엘리먼트 / 없을 경우 빈 list 반환
els = soup.select('h1.css')

In [ ]:
print('el : ', els)
print('els.text : ',[el.text for el in els])
print('els.string : ',[el.string for el in els])
print('els의 속성들 : ',[el.attrs for el in els])
print('els의 클래스 속성들 : ',[el.attrs for el in els])
print('els의 이름들 : ',[el.name for el in els])

In [ ]:
# 3_soup.find(태그, 속성) : 조건에 맞는 첫 번째 엘리먼트
print('soup.find :', soup.find('h1'),{'class':'css'})
print('soup.find :', soup.find('h1', class_='css'))
print()
print('select_one :', soup.select_one('h1#text'))
print('soup.find :', soup.find('h1',{'id':'text'}))

In [ ]:
#4 soup.find_all(태그, 속성) : 조건에 맞는 전체 엘리먼트 / 없을 경우 빈 list 반환
print('모든 h1.css 태그와 span 태그 :', soup.select('h1.css.greeting, span'))
print('모든 h1.css 태그와 span 태그 (span 태그 호출 실패) :', \
      soup.find_all(['h1', 'span'], class_=['css'])) # 태그 && 속성 ; span 출력 x
print()
print('모든 h1.css 태그와 span 태그 :', soup.find_all(['h1', 'span'], class_='css')
                                    + soup.find_all('span'))

In [ ]:
# 존재하지 않는 엘리먼트를 찾을 때의 반환값 규칙
print('find_all(empty list) :', soup.find_all('a')) # empty list : false return
print('find(None) :', soup.find('a')) # None return
print('select(empty list) :', soup.select('a')) # empty list : false return
print('select_one(None) :', soup.select_one('a')) # None return

# 2. 정적 웹 데이터 수집
- 정적 웹 크롤링

## 2.1 HTML 웹 데이터 수집
- beautifulSoup 모듈 활용

### 1) 환율 정보 가져오기
- 네이버 증권 > 시장지표 : https://finance.naver.com/marketindex/
* 크롤링 허용 범위는 사이트의 ~/robots.txt 에서 확인 가능
    - Allow : 크롤링 허용 가능
    - Disallow : 크롤링 제한 폴더

In [ ]:
# soup 객체 생성 방법 1. requests
import requests
from bs4 import BeautifulSoup

url = 'https://finance.naver.com/marketindex/'

response = requests.get(url) # Response 객체

soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
# soup 객체 생성 방법 2. urlopen
from urllib.request import urlopen

url = 'https://finance.naver.com/marketindex/'

response = urlopen(url)

soup = BeautifulSoup(response, 'html.parser')

In [ ]:
# 종류 : h3.h_lst > span.blind
# 환율 : div.head_info > span.value
# 변동률 : div.head_info > span.change
# 증감: div.head_info > span.blind

In [ ]:
#div.head_info 밑의 span.value (find 계열)
prices = []
headinfos = soup.find_all('div', class_='head_info')
for headinfo in headinfos:
    price = headinfo.find('span', class_='value')
    prices.append(float(''.join(price.text.split(','))))
print(prices)

In [ ]:
# span.value (find 계열)
price_els = soup.find_all('span', class_='value')
prices = [round(float(price.text.replace(',','')), 1) for price in price_els]
print(prices)

In [ ]:
el1 = soup.select_one('h3.h_lst > span.blind')
el2 = float(soup.select_one('div.head_info > span.value').text.replace(',',''))
el3 = soup.select_one('div.head_info > span.change')
el4 = soup.select_one('div.head_info > span.blind')

In [ ]:
print(f'{el1.text} 환율 :{el2}원 / {el3.text}% {el4.text}')

- 변수 생성 및 데이터 읽어오기
- 데이터 다듬고 사용 용도에 따라 타입 변환하기

In [ ]:
title_els = soup.select('h3.h_lst > span.blind')
unit_els = soup.select('div.head_info > span > span.blind')
trand_els = soup.select('div.head_info > span.blind')

In [ ]:
# 현재가, 변동폭
values = [round(float(el.text.replace(',','')), 1) for el in soup.select('div.head_info > span.value')]
changes = [round(float(el.text.replace(',','')), 3) for el in soup.select('div.head_info > span.change')]

In [ ]:
# 단위들 : 'div.head_info > span > span.blind'
len(unit_els) # >>> 11
units = [unit_el.string for unit_el in unit_els]
units.insert(7, '') # 단위가 없는 데이터에 '' 추가
units

In [ ]:
# 지표명, 증감
titles = [title_el.string for title_el in title_els]
trands = [trand_el.text for trand_el in trand_els]

- 데이터 프레임으로 전환하기

In [ ]:
import pandas as pd

In [ ]:
# 첫 번째 방식 (딕셔너리 리스트)
data = []
for title, value, unit, change, trand in zip(titles, values, units, changes, trands):
    # print(f'{title} : {value}{unit} / {change} - {trand}')
    data.append({
        '지표명' : title,
        '현재가' : value,
        '단위' : unit,
        '변동폭' : change,
        '증감' : trand
    })

pd.DataFrame(data)

In [ ]:
# 두 번째 방식 (2차원 리스트)
data = []
for title, value, unit, change, trand in zip(titles, values, units, changes, trands):
    data.append([title, value, unit, change, trand])
pd.DataFrame(data, columns=['title', 'value', 'unit', 'change', 'trand'])

### 2) 로또 번호 가져오기
- User Agent 추가하여 soup 생성하기
- https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto (daum의 lotto 검색 결과)

In [ ]:
# soup 객체 생성 방법 1. requests
import requests
from bs4 import BeautifulSoup

url = 'https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=lotto'

response = requests.get(url) # Response 객체

soup = BeautifulSoup(response.text, 'html.parser')

response.status_code

In [ ]:
# soup 객체 생성 방법 2. urlopen
from urllib.request import urlopen, Request

url = 'https://search.daum.net/search?w=tot&DA=YZR&t__nil_searchbox=btn&q=%ED%83%9C%ED%92%8D'
# F12 - Network - Headers - User-Agent
headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# import Request 후 url과 headers을 변수에 저장
request = Request(url, headers=headers)
# 저장한 변수로 urlopen
response = urlopen(request)

soup = BeautifulSoup(response, 'html.parser')

response.status #다음(Daum) 서버가 파이썬 기본 요청(Python-urllib)을 봇으로 감지하고 차단하여 soup 객체 정상 생성 실패

- 데이터 읽어오기

In [ ]:
episode_el = soup.select_one('span.f_red')

episode = episode_el.string

episode

In [ ]:
title_el = soup.select_one('div.prize > strong')

title = title_el.text[-4:]

title

In [ ]:
lottonum_els = soup.select('div.lottonum > span.ball:nth-last-child(n+3)')

lottonums = [lottonum.string for lottonum in lottonum_els]

lottonums

In [ ]:
bonus_el = soup.select_one('span.ball > span.screen_out')

bonus = bonus_el.string

bonus

In [ ]:
bonusnum_el = soup.select_one('div.lottonum > span.bg_ball1')

bonusnum = bonusnum_el.string

bonusnum

In [ ]:
time_el = soup.select_one('div.prize > span.date')

time = time_el.string

time

In [ ]:
print(episode, time)
print(title, [int(lottonum) for lottonum in lottonums])
print(bonus, [int(bonusnum)])

- find 계열로 변경해보기

In [ ]:
episode_el = soup.find('span', class_ ='f_red')

episode = episode_el.string

episode

In [ ]:
prize = soup.find('div', class_='prize')
title_el = prize.find('strong')

title = title_el.text[-4:]

title

In [ ]:
divlotto = soup.find('div', class_='lottonum')
lottonum_els = divlotto.find_all('span', class_='ball')[:-2]

lottonums = [lottonum.string for lottonum in lottonum_els]

lottonums

In [ ]:
txt_bonus = soup.find('span', class_='txt_bonus')
bonus_el = txt_bonus.find('span', class_='screen_out')

bonus = bonus_el.string

bonus

In [ ]:
bonusnum_el = divlotto.find('span', class_='bg_ball1')

bonusnum = bonusnum_el.string

bonusnum

In [ ]:
time_el = prize.find('span', class_='date')

time = time_el.string

time

In [ ]:
print(episode, time)
print(title, [int(lottonum) for lottonum in lottonums])
print(bonus, [int(bonusnum)])

### 3) 다음 뉴스 검색 리스트
```
no title    href
0  타이틀1  http://~
1  타이틀2  http://~
2  타이틀3  http://~
```

In [ ]:
# 방법 1
import requests
from bs4 import BeautifulSoup

word = '경주기행'
url = f'https://search.daum.net/search?w=news&q={word}'

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
# 방법 2
from urllib.request import urlopen, Request
from urllib.parse import quote

word = quote('태풍')
url = f'https://search.daum.net/search?w=news&q={word}'

# headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# request = Request(url, headers=headers)

request = Request(url)
request.add_header('User-Agent','Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')

response = urlopen(request)

soup = BeautifulSoup(response, 'html.parser')

In [ ]:
# 뉴스 링크 : div.item-title > strong.tit-g.clamp-g > a

items_find_list = []

items_el = soup.select('div.item-title > strong.tit-g.clamp-g > a')

for idx, item in enumerate(items_el):
    items_find_list.append({'no' : idx+1, 'title' : item.text, 'link' : item.attrs.get('href')})
    
rslt = items_find_list
    
import pandas as pd
pd.DataFrame(rslt)

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def collect_list(keyword, page):
    'keyword로 해당 page에 검색한 결과 list return'
    
    url = f'https://search.daum.net/search?w=news&nil_search=btn&DA=NTB&enc=utf8&cluster=y&cluster_page=1'
    params = {'q':keyword, 'p':page}
    
    headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
    response = requests.get(url, headers=headers, params=params)
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    items_find_list = []
    
    items_el = soup.select('div.item-title > strong.tit-g.clamp-g > a')

    for idx, item in enumerate(items_el):
        items_find_list.append({'no' : idx+1+(page-1)*10, 'title' : item.text, 'link' : item.attrs.get('href')})
    
    return items_find_list

In [ ]:
result = collect_list('오디세이', 1)

import pandas as pd
pd.DataFrame(result)

In [ ]:
item_result = []

pages = 2

for page in range (1, pages+1):
    print(f'== {page} 페이지 수집 중 ==')
    item_result.extend(collect_list('추석 기차 예매', page))
    time.sleep(3) # 다음 순회까지 3초 대기

result = item_result

import pandas as pd
pd.DataFrame(result)

In [ ]:
pd.DataFrame(result).set_index('no')

In [ ]:
keywords = ['인스파이어아레나', '영종도', '공연 주차 비용']
pages = 2
result0 = [] # keyword[0] 1~pages까지 검색한 결과를 dict로 반환 받는 list
result1 = [] # keyword[1] 1~pages까지 검색한 결과를 dict로 반환 받는 list
result2 = [] # keyword[2] 1~pages까지 검색한 결과를 dict로 반환 받는 list

for i, keyword in enumerate(keywords):
    print(f' == {i+1}번째 검색어 {keyword} 검색 결과 수집 중 입니다. ({page} 페이지) ==')
    for page in range (1, pages+1):
        if i==0:
            result0.extend(collect_list(keyword, page))
        elif i==1:
            result1.extend(collect_list(keyword, page))
        else:
            result2.extend(collect_list(keyword, page))
    time.sleep(3)

In [ ]:
result0_df = pd.DataFrame(result0)
result1_df = pd.DataFrame(result1)
result2_df = pd.DataFrame(result2)

In [ ]:
result0_df.head()

In [ ]:
result1_df.head()

In [ ]:
result2_df.head()

In [ ]:
result0_df.to_csv(f'data/ch14{keywords[0]}.csv', index=False, encoding ='cp949')
result1_df.to_csv(f'data/ch14{keywords[1]}.csv', index=False, encoding ='cp949')
result2_df.to_csv(f'data/ch14{keywords[2]}.csv', index=False, encoding ='cp949')

### 4) 멜론 top 100 가져오기
- User-Agent를 추가한 크롤링
- User-Agent : 접속자의 브라우저 및 운영체제 정보를 담은 헤더(신분증) 값
    - request.get(url), urlopen(url) 사용한 크롤링이 제한될 경우
    - header에 User-Agent 정보를 추가하여 사용

- melon User-Agent 정보 확인 : `https://www.melon.com/robots.txt`
    - requests.get()이나 urlopen()을 기본 상태로 사용하면 User-Agent에 'Python'으로 표시됨
    - 멜론(Melon) 등 보안이 설정된 사이트는 'Python' 헤더를 봇(Bot)으로 간주하여 접속을 차단함
    
- HTTP 상태 코드 (크롤링 시)
    - 200 (OK) : 성공
    - 403 (Forbidden) : 접근 권한 없음 (User-Agent 차단)
    - 406 (Not Acceptable) : 요청 헤더 자격 미달 (Accept, Accept-Language 등 누락)
    - 429 (Too Many Requests) : 너무 짧은 시간에 많은 요청을 보냄 (time.sleep 필요)

In [ ]:
# 방법 1
import requests
from bs4 import BeautifulSoup

url = 'https://www.melon.com/chart/'
response = requests.get(url)
response # <Response [406]> : 접근 거부로 인해 발생하는 오류

In [ ]:
# 방법 1-1 : request.get(url) 함수의 header에 User_Agent 정보를 추가하여 사용
import requests
from bs4 import BeautifulSoup

url = 'https://www.melon.com/chart/'

headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
# 방법 2 : urlopen(url) 함수의 header에 User-Agent 정보를 추가하여 사용
from urllib.request import urlopen, Request
from urllib.parse import quote

url = 'https://www.melon.com/chart/'

headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
request = Request(url, headers=headers)

response = urlopen(request)

soup = BeautifulSoup(response, 'html.parser')

In [ ]:
# 순위 : div.wrap.t_center > span.rank
# 곡명 : 
# 곡 정보 : 
# 가수 :
# 가수 정보 :

In [ ]:
rank_els = soup.select('div.wrap.t_center > span.rank')[1:] # string
songName_els = soup.select('div.ellipsis.rank01 > span > a') # string
songInfo_els = soup.select('div.wrap > a.song_info') # 'https://www.melon.com/'+[].attrs.get('href')
singerName_els = soup.select('div.ellipsis.rank02 > span.checkEllipsis') # string
singerInfo_els = soup.select('div.ellipsis.rank02 > a') # [].attrs.get('href')

In [ ]:
rank02_els = soup.select('div.ellipsis.rank02 > span.checkEllipsis')

In [ ]:
ranks = [rank.string for rank in rank_els]
songNames = [songName.string for songName in songName_els]
songLinks = ['https://www.melon.com'+songInfo.attrs.get('href') for songInfo in songInfo_els]
singerNames = [singerName.text.strip() for singerName in singerName_els]
singerLinks = [", ".join(['https://www.melon.com'+a.get('href') for a in el.select('a')]) for el in rank02_els]

In [ ]:
data = []

for rank, songName, songLink, singerName, singerLink in zip(ranks, songNames, songLinks, singerNames, singerLinks):
    
    if isinstance(singerName, list):
        singerName = ", ".join(singerName)
        
    if isinstance(singerLink, list):
        singerLink = ", ".join(singerLink)
        
    data.append({
        '순위' : rank,
        '곡명' : songName,
        '곡정보' : songLink,
        '가수명' : singerName,
        '가수정보' : singerLink
    })

In [ ]:
data[80]

In [ ]:
pd.DataFrame(data)

In [ ]:
pd.DataFrame(data).to_csv('data/ch14_멜론차트.csv')

### 5) 네이버 지식인 검색
- open API 사용 x

In [ ]:
# 방법 1
from requests import get
from bs4 import BeautifulSoup

keyword = '인스파이어 아레나'
url = f'https://kin.naver.com/search/list.naver?query={keyword}'

response = get(url)

soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
from urllib.request import urlopen, Request
from urllib.parse import quote

keyword = quote('인스파이어 아레나')
url = f'https://kin.naver.com/search/list.naver?query={keyword}'

request = Request(url)

response = urlopen(request)

soup = BeautifulSoup(response, 'html.parser')

In [ ]:
# keyword를 페이지 수만큼 크롤링

from requests import get
from bs4 import BeautifulSoup

keyword = '인스파이어아레나'
pages = 2
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}

items_list = []

for page in range(1, pages+1):
    url = f'https://kin.naver.com/search/list.naver?query={keyword}&p={page}'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # 글 제목, link
    dt_els = soup.find_all('dt')
    for dt_el in dt_els:
        item = dt_el.find('a')
        items_list.append({
            'title':item.text,
            'link':item.attrs.get('href')
        })

In [ ]:
df = pd.DataFrame(items_list)
df.T

## 2.2 open API 웹 데이터 수집
- json 웹 데이터 수집

### 네이버 지식인 검색
- open API 사용 o

**.env 파일 생성 후 client id/secret code 저장**
<br>- `%pip install dotenv`
<br>- `from dotenv iimport load_dotenv`

In [ ]:
%pip install dotenv

In [ ]:
# 환경 변수를 사용하기 위한 패키지 (.env)
from dotenv import load_dotenv
import os
load_dotenv() # id/secret 값 재할당(편집) 불가 ; 커널 초기화 시 가능
print(os.getenv('CLIENT_ID')[:3])
print(os.getenv('CLIENT_SECRET')[:3])

In [ ]:
import os
import sys
import urllib.request
import json
import pandas as pd
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("인스파이어아레나")
url = f"https://openapi.naver.com/v1/search/kin.json?query={encText}" # JSON 결과
# request = urllib.request.Request(url)
# request.add_header("X-Naver-Client-Id",client_id)
# request.add_header("X-Naver-Client-Secret",client_secret)
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
request = urllib.request.Request(url, headers=headers)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    # print(response_body.decode('utf-8')[:30])
else:
    print("Error Code:" + rescode)

data = json.loads(response_body) # 문자를 json형태로 변환
print('data의 응답들 :', data.keys())
print('검색결과 갯수 :', len(data['items']))

items = data['items']
items_list = []
for item in items:
    #print(item)
    title = item['title'].replace('<b>','').replace('</b>','')
    link  = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title','link','description'])

In [ ]:
# 네이버 검색 API 예제 - 블로그 검색 (requests/get() 사용)
import os
import requests
import json
import pandas as pd
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = "스위스"
url = f"https://openapi.naver.com/v1/search/kin.json" # JSON 결과
params = {'query':encText, 'display':20 }
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response.json()['items'] # response를 json형태로 변환한것 중 'items'
items_list = []
for item in items:
    #print(item)
    title = item['title'].replace('<b>','').replace('</b>','')
    link  = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title','link','description']).sample()

#### - NAVER API HUB 활용

In [ ]:
from dotenv import load_dotenv
import os
import requests
import json
import pandas as pd

load_dotenv() # id/secret 값 재할당(편집) 불가 ; 커널 초기화 시 가능
client_id = os.getenv('NEW_CLIENT_ID')
client_secret = os.getenv('NEW_CLIENT_SECRET')
                      
encText = "인스파이어아레나주차"
url = f"https://naverapihub.apigw.ntruss.com/search/v1/kin?query={encText}" # JSON 결과
# url = "https://openapi.naver.com/v1/search/kin.xml?query=" + encText # XML 결과

headers = {'X-NCP-APIGW-API-KEY-ID':client_id, 'X-NCP-APIGW-API-KEY':client_secret}
response = requests.get(url, headers=headers)
rescode = response.status_code
    
# data = json.loads(response.text) # 문자를 json 형태로 변환
# items = data['items']
items = response.json()['items']

items_list = []
for item in items:
    title = item['title'].replace('<b>','').replace('</b>','')
    link = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])

pd.DataFrame(items_list, columns=['title','link','description'])

### - quiz

In [ ]:
# 네이버 검색 API 예제 - 이미지 검색 (requests/urlopen() 사용)
import os
import sys
import requests
import json

client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
                    
encText = "경주천년숲"
url = f"https://openapi.naver.com/v1/search/image.json?" # JSON 결과

params = {'query':encText, 'display':100 }
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)
rescode = response.status_code

# print(response.json())

items = response.json()['items']

items_list = []
for item in items:
    title = item['title']
    link = item['link']
    thumbnail = item['thumbnail']
    sizeheight = item['sizeheight']
    sizewidth = item['sizewidth']
    items_list.append([title, link, thumbnail, sizeheight, sizewidth])

pd.DataFrame(items_list, columns=['title','link','thumbnail','sizeheight','sizewidth']).to_csv('data/img_list.csv', index=False)

### 네이버 검색 API - 이미지

In [ ]:
# 네이버 검색 API 예제 - 이미지 검색 함수 (requests/urlopen() 사용)

def get_image_list(query):
    '''
    naver api 요청을 통해 요청 데이터를 데이터 프레임으로 반환
    '''
    import os
    import sys
    import requests
    import json

    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')

    encText = f"{query}"
    url = f"https://openapi.naver.com/v1/search/image.json?" # JSON 결과

    params = {'query':encText, 'display':100 }
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    response = requests.get(url, params=params, headers=headers)

    # print(response.json())

    items = response.json()['items']

    items_list = []
    for item in items:
        title = item['title']
        link = item['link']
        thumbnail = item['thumbnail']
        sizeheight = item['sizeheight']
        sizewidth = item['sizewidth']
        items_list.append([title, link, thumbnail, sizeheight, sizewidth])

    result = pd.DataFrame(items_list, columns=['title','link','thumbnail','sizeheight','sizewidth'])
    return result

In [ ]:
df = get_image_list('천년숲')

In [ ]:
# df의 이미지 url을 이미지 로컬 파일으로 저장하기
print(df.loc[0, 'link'])
print(df.loc[4, 'thumbnail'])

In [ ]:
def save_img(attr, idx, link, query):
    'link의 이미지 url을 image/attr_idx_query.확장자로 local 저장'
    import requests
    from urllib.parse import urlparse
    
    response = requests.get(link)
    
    #link에서 확장자 추출
    # file_ext = link.split('.')[-1] # '.' 을 기준으로 분리한 것 중, 가장 마지막 요소
    file_ext = link[link.rfind('.'):] # 가장 마지막에 위치한 '.' 이후의, '.' 을 포함한 요소들
    
    # 확장자 뒤에 ?가 있는 위치
    quote_index = file_ext.find('?')
    if quote_index != -1:
        file_ext = file_ext[:quote_index] # '.' 부터 '?' 앞까지
        
    # 확장자 뒤에 %가 있는 위치
    quote_index = file_ext.find('%')
    if quote_index != -1:
        file_ext = file_ext[:quote_index] # '.' 부터 '%' 앞까지
    
    # urlparse & os.path를 통한 쿼리스트링 자동 제거
    # parsed_path = urlparse(link).path
    # file_ext = os.path.splitext(parsed_path)[1].lower()        

    # 허용할 이미지 확장자
    valid_ext = [".jpg",".jpeg",".png",".gif",".webp",".avif",".svg",".bmp",".tiff"]
    
    # 허용할 이미지 확장자가 아닐 경우, 확장자를 '.jpg'로 설정       
    file_ext = file_ext if file_ext in valid_ext else '.jpg'
    
    # link 요청을 바이너리(wb) 형식으로 변환 후 저장
    img = response.content 
    with open(f'image/{attr}_{idx:03}_{query}{file_ext}', 'wb') as f:
        f.write(img) #파일 저장

In [ ]:
save_img('메인', 0, df.loc[50, 'link'], '경주천년숲')

In [ ]:
save_img('썸네일', 0, df.loc[4, 'thumbnail'], '천년숲') # 링크에 확장자가 없는 경우

In [ ]:
# 네이버 검색 API - get_image_list ; csv - save_img ; 이미지, 썸네일 다운로드

def save_file(query):
    '''
    naver api 요청을 통해 요청 데이터를 데이터 프레임으로 반환
    '''
    import os
    import sys
    import requests
    import json

    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')

    encText = f"{query}"
    url = f"https://openapi.naver.com/v1/search/image.json?" # JSON 결과

    params = {'query':encText, 'display':100 }
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    response = requests.get(url, params=params, headers=headers)

    items = response.json()['items']

    items_list = []
    for idx, item in enumerate(items):
        title = item['title']
        link = item['link']
        thumbnail = item['thumbnail']
        sizeheight = item['sizeheight']
        sizewidth = item['sizewidth']
        items_list.append([title, link, thumbnail, sizeheight, sizewidth])
        save_img('메인', idx, link, query)
        save_img('썸네일', idx, thumbnail, query)
        if idx%20==0:
            print(f'==={idx:02}%===')
        
    result = pd.DataFrame(items_list, columns=['title','link','thumbnail','sizeheight','sizewidth'])
    result.to_csv('image/img_list.csv', index=False)
    print('이미지 및 CSV 저장 완료')
    return result

In [ ]:
df = save_file('경주천년숲')

## 2.3 XML 웹 데이터 수집
- RSS 서비스, open API 활용

### 1) 전국 날씨 RSS
- 기상청 RSS에서 1개월 간의 기상정보를 서비스 중

In [ ]:
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

item_list = []

url = 'https://www.kma.go.kr/repositary/xml/fct/mon/img/fct_mon1rss_108_20260813.xml'

response = requests.get(url)
soup = BeautifulSoup(response.text, 'xml')

locals_ = soup.select('local_ta')

for local in locals_:
    local_name = local.select_one('local_ta_name').string.strip()
    for n in range(1, 5):
        local_normalYear = local.select_one(f'week{n}_local_ta_normalYear').text
        local_similarRange = local.select_one(f'week{n}_local_ta_similarRange').text
        local_minVal = local.select_one(f'week{n}_local_ta_minVal').text
        local_similarVal = local.select_one(f'week{n}_local_ta_similarVal').text
        local_maxVal = local.select_one(f'week{n}_local_ta_maxVal').text
        item_list.append({
            '주차' : n,
            '지역' : local_name,
            '평년기온' : local_normalYear,
            '예측범위' : local_similarRange,
            '낮을확률' : local_minVal,
            '맞을확률' : local_similarVal,
            '높을확률' : local_maxVal
        })
        
df = pd.DataFrame(item_list)

In [ ]:
df['평년기온'] = df['평년기온'].astype('float')
df['낮을확률'] = df['평년기온'].astype('int')
df['맞을확률'] = df['평년기온'].astype('int')
df['높을확률'] = df['평년기온'].astype('int')

In [ ]:
df.sort_values(by=['주차', '지역']).reset_index(drop=True).head()

### 2) XML 응답 open API
- data.go.kr
    - 노선 정보 : 버스 ID, 정류장 ID와 이름
    - 버스 위치 정보 : 실시간 버스 위치 목록
    
#### 1. 버스 번호의 busRouteId 확인 ; getBusRouteList

In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from urllib.request import quote
from urllib.request import urlretrieve
import os
load_dotenv()

route_key = os.getenv('KEY')

Bus_No = 5524
busNum = quote(f'{Bus_No}')

url1 = 'http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?'
params = {'ServiceKey': route_key,'strSrch': busNum, 'resultType' : 'xml'}

# url1 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?ServiceKey={route_key}&strSrch={busNum}'
# savefilename1 = 'data/ch14_1.busInfo.xml'
# urlretrieve(url1, savefilename1)

response = requests.get(url, params=params)

soup = BeautifulSoup(response.text, 'xml')

In [ ]:
for item in soup.select('itemList'):
    busRouteNm = item.select_one('busRouteNm').text
    if busNum == busRouteNm:
        busRouteId = item.select_one('busRouteId').text
        break

print('busRouteId =', busRouteId)

#### 2. busRouteID의 정류장 목록 확인 ; getStaionsByRouteList 

In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from urllib.request import quote
from urllib.request import urlretrieve
import os
load_dotenv()

key = os.getenv('KEY')
busId = 100100260

url2 = 'http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?'
params = {'ServiceKey': key,'busRouteId': busId, 'resultType' : 'xml'}

# url2 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?ServiceKey={key}&busRouteId={busId}'
# savefilename2 = 'data/ch14_2.stationInfo.xml'
# urlretrieve(url2, savefilename2)

response = requests.get(url2, params=params)

soup = BeautifulSoup(response.text, 'xml')

itemLists = soup.select('itemList')
len(itemLists)

In [ ]:
bus_station = []

for itemList in itemLists:
    station = itemList.select_one('station').text
    stationNm = itemList.select_one('stationNm').text
    gpsX = itemList.select_one('gpsX').text
    gpsY = itemList.select_one('gpsY').text
    bus_station.append([station, stationNm, gpsX, gpsY])
    
df_station = pd.DataFrame(bus_station, columns=['ID','정류소명','경도','위도'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

df_station['경도'] = df_station['경도'].astype(float)
df_station['위도'] = df_station['위도'].astype(float)

plt.figure(figsize=(6,4))

ax=sns.scatterplot(data=df_station, x='경도', y='위도', hue='정류소명', s=10, legend=None)

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')
plt.show()

#### 3. busRouteID의 위치 정보 - getBusPosByRtidList 

In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from urllib.request import quote
from urllib.request import urlretrieve
import os
load_dotenv()

key = os.getenv('KEY')
busId = 100100260

url3 = 'http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid?'
params = {'ServiceKey': key,'busRouteId': busId, 'resultType' : 'xml'}

# url3 = f'http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid?ServiceKey={key}&busRouteId={busId}'
# savefilename3 = 'data/ch14_3.flagInfo.xml'
# urlretrieve(url3, savefilename3)

response = requests.get(url3, params=params)

soup = BeautifulSoup(response.text, 'xml')

itemLists = soup.select('itemList')
len(itemLists)

In [ ]:
bus_flag = []

for itemList in itemLists:
    congetion = itemList.select_one('congetion').text # 혼잡도(0 : 없음, 3 : 여유, 4 : 보통, 5 : 혼잡)
    congetion = '없음' if congetion=='0' else '여유' if congetion=='3' else '보통' if congetion=='4' else '혼잡'
    nextStTm = itemList.select_one('nextStTm').text
    plainNo = itemList.select_one('plainNo').text
    gpsX = itemList.select_one('gpsX').text
    gpsY = itemList.select_one('gpsY').text
    nextStId = itemList.select_one('nextStId').text
    nexStName = df_station.loc[df_station['ID']==nextStId, '정류소명'].iloc[0]
    lastStnId = itemList.select_one('lastStnId').text
    lastStName = df_station.loc[df_station['ID']==lastStnId, '정류소명'].iloc[0]
    bus_flag.append([plainNo, congetion, lastStName, nexStName, nextStTm, gpsX, gpsY])
    
df_flag = pd.DataFrame(bus_flag, columns=['차량번호','혼잡도','이전정류소','다음정류소','다음정류소도착시간','경도','위도'])
df_flag['다음정류소도착시간'] = round(df_flag['다음정류소도착시간'].astype('int')/60,2)

In [ ]:
df_flag

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

df_flag['경도'] = df_flag['경도'].astype(float)
df_flag['위도'] = df_flag['위도'].astype(float)

plt.figure(figsize=(6,4))

ax=sns.scatterplot(data=df_flag, x='경도', y='위도', hue='차량번호', s=10, legend=None)

ax.set_ylabel('위\n도', rotation=0, labelpad=15, va='center')
plt.show()

# - Quiz 1

In [ ]:
import requests
from bs4 import BeautifulSoup

pages = 2
data = []

for page in range(1,pages+1):
    url = f'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber={page}'
    savefilename = f'data/ch14_Quiz{page}.html'
    urlretrieve(url,savefilename) 
    
    response = requests.get(url, headers=headers, params=params)
    soup = BeautifulSoup(response.text, 'html.parser')

    rank_els = soup.select('div.img_upper > em.ico.rank')
    book_els = soup.select('div.info_row.info_name > a.gd_name')
    writer_els = soup.select('span.authPub.info_auth')
    publisher_els = soup.select('span.authPub.info_pub > a')
    price_els = soup.select('div.info_row.info_price > strong.txt_num > em.yes_b')

    # writer_els[1].text

    ranks = [int(rank.text) for rank in rank_els]
    books = [book.string for book in book_els]
    writers = [writer.text.strip() for writer in writer_els]
    publishers = [publisher.string for publisher in publisher_els]
    prices = [price.text for price in price_els] 
    
    for n in range(len(writers)):
        if '글' in writers[n]:
            writers[n] = writers[n][:(writers[n].find('글')+1)].strip() 
        elif '저' in writers[n]:
            writers[n] = writers[n][:(writers[n].find('저')+1)].strip()
        else:
            wrtiers[n] = wrtiers[n] 
            
    # writers[6]
    
    for rank, book, writer, publisher, price in zip(ranks, books, writers, publishers, prices):
        data.append({
            '순위' : rank,
            '책 이름' : book,
            '원저/저자/글' : writer,
            '출판사' : publisher,
            '가격' : price+'원'
        })

df = pd.DataFrame(data)
df.to_csv('ch14_yes24_bestseller.csv', index=False)   